# ANÁLISIS RESTAURANTE


*Instalar e importar librerías

In [ ]:
#!pip install pandas duckdb numpy
import pandas as pd
import numpy as np
import duckdb

Leer el archivo


In [ ]:
restaurante = pd.read_csv("../data/hotel_restaurant_orders.csv", sep = ',' , encoding="utf-8")
restaurante

* Revisión de tabla

In [ ]:
restaurante.info()

In [ ]:
restaurante.isnull().sum()

* Conexión SQL


In [ ]:
query1 = """
    SELECT 
        DATE_TRUNC('MONTH', CAST(OrderDate AS DATE)) AS mes,
        COUNT(DISTINCT OrderID) AS total_ordenes
    FROM 
        restaurante
    GROUP BY 
        DATE_TRUNC('MONTH', CAST(OrderDate AS DATE))
    ORDER BY
        mes ASC;
"""

In [ ]:
duckdb.sql(query1).df()


Las órdenes se distribuyeron de forma estable durante 2025, con un rango entre 289 y 315 órdenes mensuales. Octubre fue el mes con mayor actividad (315 órdenes) y septiembre el de menor actividad (289 órdenes).


In [ ]:
query2 = """
    SELECT
        CustomerID AS cliente_id,
        COUNT(DISTINCT OrderID) AS cantidad_ordenes
    FROM restaurante
    GROUP BY cliente_id
    ORDER BY cantidad_ordenes DESC;
"""

In [ ]:
duckdb.sql(query2).df()


* Registros únicos


In [ ]:
query3 = """
    SELECT
        COUNT(*)
    FROM restaurante;
"""

duckdb.sql(query3).df()


* Revisión de duplicados

In [ ]:
query4 = """
WITH duplicados AS 
    (SELECT
        DISTINCT * 
    FROM restaurante
)

    SELECT
        COUNT(*)
    FROM duplicados;

"""

duckdb.sql(query4).df()

In [ ]:
query5 = """
SELECT MenuCategory,
    COUNT (*) AS cantidad_ordenes 
FROM restaurante
GROUP BY MenuCategory
ORDER BY cantidad_ordenes DESC;
"""

duckdb.sql(query5).df()

La categoría de bebidas presenta el mayor número de órdenes, con 1.914 pedidos, seguida por alimentos con 1.729. La diferencia es moderada, por lo que ambas categorías tienen una participación relevante en las ventas.
 

In [ ]:
query6 = """
SELECT RestaurantType, 
    COUNT (*) AS cantidad_ordenes 
FROM restaurante
GROUP BY RestaurantType
ORDER BY cantidad_ordenes DESC;
"""

duckdb.sql(query6).df()

El consumo dentro del establecimiento predomina con 1.757 órdenes. Los domicilios ocupan el segundo lugar con 1.121 órdenes y los pedidos para llevar suman 765, por lo que la experiencia presencial continúa siendo el principal canal de consumo.



In [ ]:
query7 = """
SELECT ItemName, 
    COUNT (*) AS cantidad_ordenes 
FROM restaurante
GROUP BY ItemName
ORDER BY cantidad_ordenes DESC;
"""

duckdb.sql(query7).df()

Los productos con más órdenes son el té helado (350), el capuchino (347) y la pizza margarita (328). Entre los alimentos también destacan la ensalada César (276) y el pollo a la parrilla (275), lo que evidencia una demanda repartida entre bebidas y opciones de comida.



### Segmentación de clientes según su actividad con el Hotel

En esta sección se segmentan los clientes en tres grupos de valor para el hotel durante el último año. Con las reglas actuales, se obtienen 80 huéspedes VIP, 150 huéspedes conectados y 250 huéspedes esporádicos.

* Huéspedes VIP: han asistido durante 8 o más meses, han realizado 14 o más órdenes y han consumido 300 dólares o más.
* Huéspedes conectados: han asistido durante 4 o más meses, han realizado 6 o más órdenes y han consumido 150 dólares o más, sin cumplir las condiciones de VIP.
* Huéspedes esporádicos: no cumplen simultáneamente las condiciones definidas para los segmentos anteriores. 

In [ ]:
query8 = """
WITH agregados AS (
    SELECT 
        CustomerID,
        COUNT(DISTINCT DATE_PART('MONTH', CAST(OrderDate AS DATE))) AS recurrencia,
        COUNT(DISTINCT OrderID) AS cantidad_ordenes,
        SUM(TotalPrice) AS valor_total 
    FROM 
        restaurante
    GROUP BY 
        CustomerID
)

SELECT  
    CustomerID,
    CASE
        WHEN recurrencia >=8 AND cantidad_ordenes >=14 AND valor_total >=300 THEN 'Huespedes VIP' 
        WHEN recurrencia >=4 AND cantidad_ordenes >=6 AND valor_total >=150 THEN 'Huespedes conectados' 
        ELSE 'Huespedes esporádicos'
    END AS segmento_valor
FROM 
    agregados
"""


clientes =duckdb.sql(query8).df()
clientes

In [ ]:
query9 = """

SELECT  
    segmento_valor,
    COUNT(DISTINCT CustomerID) AS cantidad_clientes
FROM 
    clientes
GROUP BY 
    segmento_valor
"""


conteo_clientes =duckdb.sql(query9).df()
conteo_clientes

* Exportar la base de clientes

In [ ]:
clientes.to_csv("../data/segmentos_clientes.csv", index=False, sep=',', encoding='utf-8')

### Limpiar tabla de datos transaccional 

In [ ]:
query10 = """

SELECT  
    OrderID, 
    CustomerID,	
    CAST(OrderDate AS DATE) AS OrderDate,
    RestaurantType,	
    MenuCategory,	
    ItemName,	
    Quantity,	
    UnitPrice,	
    TotalPrice,	
    PaymentMethod,	
    CASE
        WHEN TableNumber = '-' THEN 'N/A'
        ELSE TableNumber
    END AS TableNumber,
    DayOfWeek,	
    TimeOfDay,	
    CASE
        WHEN SpecialRequest IS NULL THEN 'None'
        ELSE SpecialRequest
    END AS SpecialRequest,
    CASE
        WHEN ServerName = '-' THEN 'does not report' 
        ELSE ServerName
    END AS ServerName,

FROM 
    restaurante
"""


transacciones =duckdb.sql(query10).df()
transacciones

In [ ]:
transacciones.to_csv("../data/transacciones.csv", index=False, sep=',', encoding='utf-8')